<img src="../assets/tumor_twin.png" alt="Tumor Twin" width="500"/>

# Coupled PDE system demo (immune + tumor)

This notebook mirrors the **HGG_Demo** workflow—data → model → solver → prediction → comparison → calibration—but for a **stacked PDE state** `(C, D, H, W)` using `ImmuneResponse3D` (tumor + lymphocytes).

**Key idea:** `TorchDiffEqSolver.solve` returns `u` with shape `(T, C, D, H, W)`. Imaging and calibration usually target **one component** (tumor = channel `0`). Use `tumortwin.pde_workflow` to extract tumor maps and build LM residuals like the single-field tutorial.

---
## Table of contents
- [Step 1: Build model & initial state](#step-1)
- [Step 2: Forward solve](#step-2)
- [Step 3: Postprocess (tumor-only maps)](#step-3)
- [Step 4: Calibration residual (LM-ready)](#step-4)
---

In [ ]:
from datetime import datetime, timedelta
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
import torch

from tumortwin.models.immune_3d import ImmuneResponse3D
from tumortwin.pde_workflow import (
    fields_at_times_from_trajectory,
    initial_pde_state_from_tumor_field,
    select_timepoint_indices,
    spatiotemporal_residual_vector,
    squared_error_loss,
    trajectory_to_map_list,
)
from tumortwin.postprocessing.prediction_summary import (
    plot_cellularity_map,
    plot_predicted_TCC,
)
from tumortwin.solvers import TorchDiffEqSolver, TorchDiffEqSolverOptions
from tumortwin.types.imaging import NibabelNifti
from tumortwin.utils import days_since_first

## Step 1: Build model & initial state

We use a **small synthetic volume** so the notebook runs quickly without patient JSON files. On real data, replace the stub with `HGGPatientData.from_file(...)` (or TNBC) and set `initial_u1` from `ADC_to_cellularity` like **HGG_Demo**.

If you only have a tumor map `(D,H,W)`, build a stacked IC with `initial_pde_state_from_tumor_field(..., other_fill=1.0)` for the second species—or call `model.get_initial_state()` after constructing `ImmuneResponse3D`.

In [ ]:
shape = (10, 12, 8)
mask = np.ones(shape, dtype=np.float32)
brain = NibabelNifti.from_array(mask)
t1 = NibabelNifti.from_array(np.random.rand(*shape).astype(np.float32))
patient_data = SimpleNamespace(brainmask_image=brain, T1_post_image=t1)

tumor_ic = torch.rand(shape, dtype=torch.float32) * 0.25
u0 = initial_pde_state_from_tumor_field(
    tumor_ic, num_components=2, other_fill=1.0
)

t0 = datetime(2024, 1, 1)
model = ImmuneResponse3D(
    D1=torch.tensor(0.02),
    mu1=torch.tensor(0.04),
    gamma12=torch.tensor(0.02),
    D4=torch.tensor(0.03),
    gamma21=torch.tensor(0.02),
    v=[0.0, 0.0, 0.0],
    patient_data=patient_data,
    initial_time=t0,
    initial_u1=tumor_ic,
    radiotherapy_specification=None,
    chemotherapy_specifications=None,
    require_grad=True,
    device=torch.device("cpu"),
)

u0_model = model.get_initial_state()
assert u0_model.shape == (2,) + shape
torch.testing.assert_close(u0, u0_model)

## Step 2: Forward solve

Same API as **HGG_Demo** Step 3–4: `TorchDiffEqSolver` + `solve(timepoints, u_initial)`.
Initial condition must match the model (`(C,D,H,W)` here).

In [ ]:
solver = TorchDiffEqSolver(
    model,
    TorchDiffEqSolverOptions(
        step_size=timedelta(days=1.0),
        method="rk4",
        device=torch.device("cpu"),
        use_adjoint=True,
    ),
)

timepoints = [t0 + timedelta(days=float(d)) for d in (0, 5, 10, 20)]
t_days, trajectory = solver.solve(timepoints=timepoints, u_initial=u0)
print("t shape", t_days.shape, "u shape", tuple(trajectory.shape))

## Step 3: Postprocess (tumor-only maps)

`plot_predicted_TCC` and `plot_cellularity_map` expect **one** field per time, shape `(D,H,W)`. Use `trajectory_to_map_list(trajectory, component_idx=0)`.

In [ ]:
tumor_maps = trajectory_to_map_list(trajectory, component_idx=0)
carrying_capacity = 1.0e6

fig, ax = plt.subplots(1, 1, figsize=(5, 3))
plot_predicted_TCC(tumor_maps, timepoints, ax=ax, carrying_capacity=carrying_capacity)
ax.set_title("Tumor TCC (component 0)")
plt.show()

fig, axes = plt.subplots(1, len(tumor_maps), figsize=(3 * len(tumor_maps), 3))
if len(tumor_maps) == 1:
    axes = [axes]
for i, (u_t, tp) in enumerate(zip(tumor_maps, timepoints)):
    plot_cellularity_map(
        u_t, patient_data, time=float(days_since_first(tp, timepoints[0])), ax=axes[i]
    )
plt.tight_layout()
plt.show()

## Step 4: Calibration residual (LM-ready)

**HGG_Demo** Step 7 compares predicted and measured cellularity at selected visits. For PDE output, pick time rows with `select_timepoint_indices`, extract tumor maps with `fields_at_times_from_trajectory`, then flatten differences with `spatiotemporal_residual_vector`. Pass a vector-valued closure to `LMoptimizer` (same as the tutorial).

In [ ]:
visit_days = [0.0, 10.0]
idx = select_timepoint_indices(t_days, visit_days, atol=0.05)
pred_at_visits = fields_at_times_from_trajectory(trajectory, idx, component_idx=0)

measured = [(tumor_maps[i] * 0.95).detach() for i in idx]
res = spatiotemporal_residual_vector(pred_at_visits, measured)
loss = squared_error_loss(res)
print("Residual size", res.numel(), "SSE", float(loss))

# Example: scalar loss for autograd (tune D1, etc.)
model.zero_grad(set_to_none=True)
loss.backward()
print("d(loss)/d(D1)", model.D1.grad)

## Conclusion

- **Solver:** unchanged; pass `(C,D,H,W)` initial state from `get_initial_state()` or `initial_pde_state_from_tumor_field`.
- **Plots / TCC:** convert trajectory with `trajectory_to_map_list(..., 0)`.
- **Calibration:** build residuals on tumor maps only via `fields_at_times_from_trajectory` + `spatiotemporal_residual_vector`.

See **docs → API → Coupled PDE systems & workflow** for full reference.